# Load dataset

In [2]:
from __future__ import annotations

import os
import re
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["axes.grid"] = True

TRAIN_ROOT = Path("../../datasets/param_controller/training")
VAL_ROOT = Path("../../datasets/param_controller/validation")

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

ZERO_VAR_STD = 1e-8
NEAR_ZERO_VAR_STD = 1e-4
HIGH_CORR_THRESHOLD = 0.95
STRONG_CORR_THRESHOLD = 0.85

FEATURE_COLS = None

In [3]:
def discover_files(root: Path):
    if not root.exists():
        raise FileNotFoundError(f"Dataset folder not found: {root}")
    files = sorted(
        p for p in root.rglob("*")
        if p.is_file() and p.suffix.lower() in {".npz", ".pkl", ".pickle"}
    )
    if not files:
        raise ValueError(f"No dataset files found under {root}")
    return files

def parse_file_metadata(path: Path):
    parts = path.parts
    split = "unknown"
    size = "unknown"

    if "training" in parts:
        split = "training"
        idx = parts.index("training")
        if idx + 1 < len(parts):
            size = parts[idx + 1]
    elif "validation" in parts:
        split = "validation"
        idx = parts.index("validation")
        if idx + 1 < len(parts):
            size = parts[idx + 1]

    instance = path.stem
    run_id = "unknown"
    m = re.search(r"(.+)_run(\d+)$", path.stem)
    if m:
        instance = m.group(1)
        run_id = int(m.group(2))

    return {
        "split": split,
        "size": size,
        "instance": instance,
        "run_id": run_id,
        "file_name": path.name,
        "file_path": str(path),
    }

def load_dataset_file(path: Path):
    meta = parse_file_metadata(path)

    # Try pickle payload first (this is how OfflineDataset.save() currently writes files).
    try:
        with open(path, "rb") as f:
            payload = pickle.load(f)

        if isinstance(payload, dict) and "data" in payload and "feature_names" in payload:
            data = payload["data"]
            feature_names = list(payload["feature_names"])
            meta["format"] = "pickle_payload"
            meta["normalized"] = payload.get("normalized", False)

            states = np.asarray(data["states"], dtype=np.float32)
            actions = np.asarray(data["actions"], dtype=np.int64)
            rewards = np.asarray(data["rewards"], dtype=np.float32)
            next_states = np.asarray(data["next_states"], dtype=np.float32)
            dones = np.asarray(data["dones"], dtype=np.bool_)

            meta_block = data.get("meta", {})
            meta_arrays = {}
            if isinstance(meta_block, dict):
                for k, v in meta_block.items():
                    meta_arrays[k] = np.asarray(v)
            return feature_names, states, actions, rewards, next_states, dones, meta, meta_arrays
    except Exception:
        pass

    # Fallback: true npz archive.
    with np.load(path, allow_pickle=True) as npz:
        meta["format"] = "npz"
        meta["normalized"] = None
        feature_names = list(npz["feature_names"].tolist()) if "feature_names" in npz.files else None
        states = np.asarray(npz["states"], dtype=np.float32)
        actions = np.asarray(npz["actions"], dtype=np.int64)
        rewards = np.asarray(npz["rewards"], dtype=np.float32)
        next_states = np.asarray(npz["next_states"], dtype=np.float32)
        dones = np.asarray(npz["dones"], dtype=np.bool_)
        meta_arrays = {}
        if "meta" in npz.files:
            meta_raw = npz["meta"].item()
            if isinstance(meta_raw, dict):
                for k, v in meta_raw.items():
                    meta_arrays[k] = np.asarray(v)
        return feature_names, states, actions, rewards, next_states, dones, meta, meta_arrays

def build_transition_dataframe(root: Path):
    files = discover_files(root)
    dfs = []
    file_summaries = []
    feature_names_global = None

    for fp in files:
        feature_names, states, actions, rewards, next_states, dones, meta, meta_arrays = load_dataset_file(fp)

        if feature_names is not None:
            if feature_names_global is None:
                feature_names_global = feature_names
            elif feature_names != feature_names_global:
                raise ValueError(
                    f"Feature mismatch in file {fp}\n"
                    f"Expected: {feature_names_global}\n"
                    f"Got: {feature_names}"
                )

        if feature_names_global is None:
            raise ValueError(f"No feature_names found in {fp}")

        n = len(actions)
        if not (states.shape[0] == n == rewards.shape[0] == next_states.shape[0] == dones.shape[0]):
            raise ValueError(f"Length mismatch in file {fp}")

        df = pd.DataFrame(states, columns=feature_names_global)
        for i, name in enumerate(feature_names_global):
            df[f"next_{name}"] = next_states[:, i]

        df["action"] = actions
        df["reward"] = rewards
        df["done"] = dones.astype(int)

        for k, v in meta.items():
            df[k] = v

        for k, arr in meta_arrays.items():
            if len(arr) == n:
                df[f"meta_{k}"] = arr
            else:
                df[f"meta_{k}"] = [repr(x) for x in arr]

        df["trans_id"] = np.arange(n)
        dfs.append(df)

        # per-file summary
        counts = np.bincount(actions.astype(int), minlength=int(actions.max()) + 1)
        p = counts / counts.sum()
        action_entropy = float(-(np.where(p > 0, p * np.log(p), 0.0)).sum())

        file_summaries.append({
            "file": fp.name,
            "split": meta["split"],
            "size": meta["size"],
            "instance": meta["instance"],
            "run_id": meta["run_id"],
            "transitions": n,
            "reward_mean": float(np.mean(rewards)),
            "reward_std": float(np.std(rewards)),
            "reward_min": float(np.min(rewards)),
            "reward_max": float(np.max(rewards)),
            "action_entropy": action_entropy,
            "format": meta.get("format"),
            "normalized": meta.get("normalized"),
        })

    full_df = pd.concat(dfs, ignore_index=True)
    file_summary_df = pd.DataFrame(file_summaries)
    return full_df, file_summary_df, feature_names_global, files

In [4]:
train_df, train_file_summary, FEATURE_COLS, train_files = build_transition_dataframe(TRAIN_ROOT)
val_df, val_file_summary, _, val_files = build_transition_dataframe(VAL_ROOT)

print("Training files:", len(train_files))
print("Validation files:", len(val_files))
print("Feature count:", len(FEATURE_COLS))
print("Training transitions:", len(train_df))
print("Validation transitions:", len(val_df))

Training files: 56
Validation files: 20
Feature count: 40
Training transitions: 148265
Validation transitions: 68997


# Cleaning rules

## Feature cleaning config

In [5]:
ZERO_VAR_STD = 1e-8
NEAR_ZERO_VAR_STD = 1e-4
HIGH_CORR_THRESHOLD = 0.95

# 1) Hard drop 
DROP_FEATURES_MANUAL = [
    # exact duplicates
    "k_norm",
    "neigh_id",
    "time_left_ratio",
    "ls_fail_ratio_W",

    # strong redundancy
    "penT_ratio",
    "n_customers",
    "max_delta_W",
    "time_elapsed_ratio",

    # dead features
    "avg_delta_W_norm",
    "sync_ratio",
    "max_delta_W_norm",
    "n_vehicles",
    "stagnation_ls",
    "early_tw_ratio",   
    "penS_ratio",
    "param_stagnation_limit",
    "tw_violation_ratio",
]

# 2) Heavy-tailed features (need log scaling)
LOG_SCALE_FEATURES = [
    "delta_trend",
    "avg_delta_W",
    "pen_ratio",
    "avg_lateness_norm",
]

# 3) Features to KEEP as reference representatives
PREFERRED_FEATURES = {
    "k": ["k_norm", "neigh_id"],
    "elapsed": ["time_elapsed_ratio", "time_left_ratio"],
    "improv_rate_W": ["ls_fail_ratio_W"],
    "pen_ratio": ["penT_ratio"],
    "avg_delta_W": ["max_delta_W"],
    "PV": ["n_customers"],
}

## Auto-detect extra useless features

In [6]:
def detect_low_information_features(df, feature_cols):
    summary = []

    for c in feature_cols:
        s = pd.to_numeric(df[c], errors="coerce")
        summary.append({
            "feature": c,
            "std": s.std(),
            "unique": s.nunique(dropna=True),
            "pct_zero": float((s == 0).mean()),
        })

    summary_df = pd.DataFrame(summary)

    dead = summary_df[
        (summary_df["std"].fillna(0) <= ZERO_VAR_STD) |
        (summary_df["unique"] <= 1)
    ]["feature"].tolist()

    near_dead = summary_df[
        (summary_df["std"].fillna(0) <= NEAR_ZERO_VAR_STD) |
        (summary_df["pct_zero"] >= 0.99)
    ]["feature"].tolist()

    return dead, near_dead, summary_df

## cleaning function

In [7]:
from sklearn.preprocessing import RobustScaler

def clean_dataset(df: pd.DataFrame, feature_cols: list):
    df = df.copy()

    print("Initial shape:", df.shape)

    # =========================
    # 1. Drop manual bad features
    # =========================
    drop_cols = [c for c in DROP_FEATURES_MANUAL if c in df.columns]
    print(f"Dropping {len(drop_cols)} manual features:", drop_cols)

    df = df.drop(columns=drop_cols, errors="ignore")

    # Update feature cols
    feature_cols = [c for c in feature_cols if c not in drop_cols]

    # =========================
    # 2. Auto-detect dead features
    # =========================
    dead, near_dead, summary_df = detect_low_information_features(df, feature_cols)

    print(f"Auto-detected dead features: {dead}")

    df = df.drop(columns=dead, errors="ignore")
    feature_cols = [c for c in feature_cols if c not in dead]

    # =========================
    # 3. Log scaling (heavy tails)
    # =========================
    for c in LOG_SCALE_FEATURES:
        if c in df.columns:
            print(f"Log-scaling {c}")
            df[c] = np.sign(df[c]) * np.log1p(np.abs(df[c]))

    # =========================
    # 4. Robust normalization
    # =========================
    scaler = RobustScaler()

    print("Applying RobustScaler...")
    df[feature_cols] = scaler.fit_transform(df[feature_cols])

    print("Final shape:", df.shape)

    return df, feature_cols, scaler

# Applying cleaning

In [8]:
# Clean training
train_clean, CLEAN_FEATURE_COLS, scaler = clean_dataset(train_df, FEATURE_COLS)

# Clean validation (same pipeline BUT no fit)
def apply_clean_pipeline(df, feature_cols, scaler):
    df = df.copy()

    # 1. Drop same features
    drop_cols = [c for c in DROP_FEATURES_MANUAL if c in df.columns]
    df = df.drop(columns=drop_cols, errors="ignore")

    feature_cols = [c for c in feature_cols if c not in drop_cols]

    # 2. Drop dead features (same as training)
    dead, _, _ = detect_low_information_features(df, feature_cols)
    df = df.drop(columns=dead, errors="ignore")
    feature_cols = [c for c in feature_cols if c not in dead]

    # 3. Log scaling
    for c in LOG_SCALE_FEATURES:
        if c in df.columns:
            df[c] = np.sign(df[c]) * np.log1p(np.abs(df[c]))

    # 4. Apply SAME scaler
    df[feature_cols] = scaler.transform(df[feature_cols])

    return df

val_clean = apply_clean_pipeline(val_df, CLEAN_FEATURE_COLS, scaler)

Initial shape: (148265, 101)
Dropping 17 manual features: ['k_norm', 'neigh_id', 'time_left_ratio', 'ls_fail_ratio_W', 'penT_ratio', 'n_customers', 'max_delta_W', 'time_elapsed_ratio', 'avg_delta_W_norm', 'sync_ratio', 'max_delta_W_norm', 'n_vehicles', 'stagnation_ls', 'early_tw_ratio', 'penS_ratio', 'param_stagnation_limit', 'tw_violation_ratio']
Auto-detected dead features: []
Log-scaling delta_trend
Log-scaling avg_delta_W
Log-scaling pen_ratio
Log-scaling avg_lateness_norm
Applying RobustScaler...
Final shape: (148265, 84)


In [9]:
# consistency check
print("Train features:", len(CLEAN_FEATURE_COLS))
print("Val features:", len(CLEAN_FEATURE_COLS))

assert all(c in val_clean.columns for c in CLEAN_FEATURE_COLS)

Train features: 23
Val features: 23


# Save cleaned dataset

In [10]:
OUT_DIR = Path("../../datasets/param_controller_cleaned")
OUT_DIR.mkdir(parents=True, exist_ok=True)

def save_numpy_dataset(df, feature_cols, path):
    np.savez_compressed(
        path,
        states=df[feature_cols].values.astype(np.float32),
        actions=df["action"].values.astype(np.int64),
        rewards=df["reward"].values.astype(np.float32),
        dones=df["done"].values.astype(np.int8),
        feature_names=np.array(feature_cols)
    )

save_numpy_dataset(train_clean, CLEAN_FEATURE_COLS, OUT_DIR / "train_clean.npz")
save_numpy_dataset(val_clean, CLEAN_FEATURE_COLS, OUT_DIR / "val_clean.npz")